# Этап 9 V1 — OOF rank-capacity / blind-spot rescue

## Исследовательский вопрос

При одинаковой фиксированной high-risk capacity действительно ли **FT-Transformer** лучше `GBDT_mean` захватывает заранее определённые **805 дефолтов Stage 3 blind spot**?

Нового обучения нет: используются только принятые row-level OOF probabilities.

## Почему проверяем сейчас

В Stage 8 V1 FT-Transformer имел более высокий Recall при диагностическом threshold `0.5`, но уступил `GBDT_mean` по OOF Gini. Этот анализ отделяет возможное улучшение ранжирования трудных дефолтов от эффекта иной шкалы score.

**Неизменно:** working sample, target, Stage 3 blind spot, две модели, OOF-only и закрытый final test.  
**Меняется:** вместо threshold используется exact top-K при фиксированной capacity.

## Запуск notebook

Notebook не предполагает, что Jupyter/VS Code запущен из корня проекта: ROOT детерминированно ищется вверх от текущего `cwd` по `pyproject.toml` и `reports/generated`. Поэтому запуск поддерживается как из корня проекта, так и из `notebooks/`; все artifact paths строятся от найденного ROOT.


In [2]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the nearest project root from cwd without assuming Jupyter's cwd."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'reports' / 'generated').is_dir():
            return candidate
    raise RuntimeError(
        'Не найден корень проекта: проверены cwd и его parents на pyproject.toml '
        'и структуру reports/generated.'
    )


ROOT = find_project_root(Path.cwd())
GENERATED = ROOT / 'reports' / 'generated'
SUMMARY = ROOT / 'reports' / 'summary'
STAGE3_OOF = GENERATED / 'stage3_oof_predictions_V1.npz'
STAGE7_OOF = GENERATED / 'stage7_tabm_stacking_oof_V1.npz'
STAGE8_OOF = GENERATED / 'stage8_ft_transformer_oof_V1.npz'
STAGE3_RESULTS = GENERATED / 'stage3_error_analysis_results_V1.json'
EXPECTED_N = 289_614
EXPECTED_WORKING_SHA256 = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
BLIND_SPOT_N = 805
CAPACITIES = {10: 28_961, 20: 57_922, 30: 86_884}

print(f'Project ROOT: {ROOT}')


Project ROOT: D:\Projects\komus-work


In [3]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def sha256_array(values: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()

stage3_diagnostics = json.loads(STAGE3_RESULTS.read_text(encoding='utf-8'))
accepted_rules = stage3_diagnostics['диагностические_границы']
deep_miss_threshold = float(
    accepted_rules['глубоко_пропущенный_дефолт_consensus_rank_lte']
)
blind_spot_spread_threshold = float(
    accepted_rules['общая_слепая_зона_rank_spread_lte']
)
assert (deep_miss_threshold, blind_spot_spread_threshold) == (0.50, 0.10)

with np.load(STAGE3_OOF, allow_pickle=False) as stage3, \
     np.load(STAGE7_OOF, allow_pickle=False) as stage7, \
     np.load(STAGE8_OOF, allow_pickle=False) as stage8:
    required_stage3 = {'working_indices', 'target', 'oof_catboost', 'oof_xgboost', 'oof_lightgbm', 'consensus_rank', 'rank_spread'}
    required_stage7 = {'working_indices', 'target', 'gbdt_mean'}
    required_stage8 = {'working_indices', 'target', 'ft_transformer_oof_probability', 'gbdt_mean_probability'}
    if not required_stage3.issubset(stage3.files) or not required_stage7.issubset(stage7.files) or not required_stage8.issubset(stage8.files):
        raise RuntimeError('Один из accepted OOF artifacts не содержит обязательные массивы.')

    working_indices = np.asarray(stage3['working_indices'], dtype=np.int64)
    target = np.asarray(stage3['target'], dtype=np.int8)
    working_indices_7 = np.asarray(stage7['working_indices'], dtype=np.int64)
    target_7 = np.asarray(stage7['target'], dtype=np.int8)
    working_indices_8 = np.asarray(stage8['working_indices'], dtype=np.int64)
    target_8 = np.asarray(stage8['target'], dtype=np.int8)
    gbdt_mean = np.asarray(stage7['gbdt_mean'], dtype=np.float64)
    ft_transformer = np.asarray(stage8['ft_transformer_oof_probability'], dtype=np.float64)
    gbdt_mean_stage8 = np.asarray(stage8['gbdt_mean_probability'], dtype=np.float64)
    stage3_probabilities = [np.asarray(stage3[name], dtype=np.float64) for name in ('oof_catboost', 'oof_xgboost', 'oof_lightgbm')]
    consensus_rank = np.asarray(stage3['consensus_rank'], dtype=np.float64)
    rank_spread = np.asarray(stage3['rank_spread'], dtype=np.float64)

assert len(working_indices) == len(target) == len(gbdt_mean) == len(ft_transformer) == EXPECTED_N
assert np.array_equal(working_indices, working_indices_7) and np.array_equal(working_indices, working_indices_8)
assert np.array_equal(target, target_7) and np.array_equal(target, target_8)
assert sha256_array(working_indices) == EXPECTED_WORKING_SHA256
assert all(np.isfinite(values).all() for values in [*stage3_probabilities, gbdt_mean, ft_transformer, gbdt_mean_stage8])
assert np.array_equal(gbdt_mean, gbdt_mean_stage8)

preflight = {
    'n': int(len(working_indices)),
    'working_indices_exact_equal': True,
    'target_exact_equal': True,
    'working_index_sha256': sha256_array(working_indices),
    'probability_arrays_finite': True,
    'gbdt_mean_stage7_stage8_exact_equal': True,
    'accepted_stage3_thresholds': {
        'consensus_rank_lte': deep_miss_threshold,
        'rank_spread_lte': blind_spot_spread_threshold,
    },
}
display(pd.DataFrame([preflight]))


,n,working_indices_exact_equal,target_exact_equal,working_index_sha256,probability_arrays_finite,gbdt_mean_stage7_stage8_exact_equal
0,289614,True,True,80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d...,True,True


In [4]:
# Blind spot воспроизводится только по accepted Stage 3 rule из diagnostics JSON.
blind_spot = (target == 1) & (consensus_rank <= deep_miss_threshold) & (rank_spread <= blind_spot_spread_threshold)
assert int(blind_spot.sum()) == BLIND_SPOT_N
assert bool(np.all(target[blind_spot] == 1))

blind_spot_working_indices = np.asarray(working_indices[blind_spot], dtype=np.int64)
assert np.array_equal(blind_spot_working_indices, working_indices_7[blind_spot])
assert np.array_equal(blind_spot_working_indices, working_indices_8[blind_spot])
blind_spot_cohort_sha256 = sha256_array(blind_spot_working_indices)

print(f'Stage 3 blind spot: {blind_spot.sum()} строк; все имеют target == 1.')
print(f'blind_spot_cohort_sha256: {blind_spot_cohort_sha256}')


Stage 3 blind spot: 805 строк; все имеют target == 1.


In [5]:
def exact_top_k(probability: np.ndarray, k: int) -> np.ndarray:
    """Probability descending; ties break by working_index ascending."""
    order = np.lexsort((working_indices, -np.asarray(probability, dtype=np.float64)))
    selected = np.zeros(len(probability), dtype=bool)
    selected[order[:k]] = True
    assert int(selected.sum()) == k
    return selected

default_count = int(target.sum())
rows, selected_by_capacity = [], {}
for capacity_pct, k in CAPACITIES.items():
    selected_gbdt = exact_top_k(gbdt_mean, k)
    selected_ft = exact_top_k(ft_transformer, k)
    selected_by_capacity[capacity_pct] = (selected_gbdt, selected_ft)
    rows.append({
        'Capacity': f'{capacity_pct}%', 'K': k,
        'GBDT_mean rescue, n': int((blind_spot & selected_gbdt).sum()),
        'GBDT_mean rescue, %': 100 * (blind_spot & selected_gbdt).sum() / BLIND_SPOT_N,
        'FT-Transformer rescue, n': int((blind_spot & selected_ft).sum()),
        'FT-Transformer rescue, %': 100 * (blind_spot & selected_ft).sum() / BLIND_SPOT_N,
        'GBDT_mean default capture, %': 100 * (target.astype(bool) & selected_gbdt).sum() / default_count,
        'FT-Transformer default capture, %': 100 * (target.astype(bool) & selected_ft).sum() / default_count,
    })
table = pd.DataFrame(rows)
display(table.round(3))


,Capacity,K,"GBDT_mean rescue, n","GBDT_mean rescue, %","FT-Transformer rescue, n","FT-Transformer rescue, %","GBDT_mean default capture, %","FT-Transformer default capture, %"
0,10%,28961,0,0.0,0,0.000,57.416,56.973
1,20%,57922,0,0.0,0,0.000,78.044,77.890
2,30%,86884,0,0.0,9,1.118,87.475,87.435


In [6]:
selected_gbdt_30, selected_ft_30 = selected_by_capacity[30]
decomposition = {
    'both': int((blind_spot & selected_gbdt_30 & selected_ft_30).sum()),
    'FT-only': int((blind_spot & ~selected_gbdt_30 & selected_ft_30).sum()),
    'GBDT-only': int((blind_spot & selected_gbdt_30 & ~selected_ft_30).sum()),
    'neither': int((blind_spot & ~selected_gbdt_30 & ~selected_ft_30).sum()),
}
assert sum(decomposition.values()) == BLIND_SPOT_N
display(pd.DataFrame([decomposition]))

delta_rescue_30_pp = float(table.loc[table['Capacity'] == '30%', 'FT-Transformer rescue, %'].iloc[0] - table.loc[table['Capacity'] == '30%', 'GBDT_mean rescue, %'].iloc[0])
decision = 'material_rank_complementarity' if delta_rescue_30_pp >= 5.0 else 'no_material_rank_complementarity'
print(f'Primary Δ rescue @30%: {delta_rescue_30_pp:.3f} п.п.')
print(f'Decision: {decision}')


,both,FT-only,GBDT-only,neither
0,0,9,0,796


Primary Δ rescue @30%: 1.118 п.п.
Decision: no_material_rank_complementarity


In [7]:
artifact_identities = {
    'stage3_oof_predictions': {'path': str(STAGE3_OOF.relative_to(ROOT)), 'sha256': sha256_file(STAGE3_OOF)},
    'stage7_tabm_stacking_oof': {'path': str(STAGE7_OOF.relative_to(ROOT)), 'sha256': sha256_file(STAGE7_OOF)},
    'stage8_ft_transformer_oof': {'path': str(STAGE8_OOF.relative_to(ROOT)), 'sha256': sha256_file(STAGE8_OOF)},
    'stage3_error_analysis_results': {'path': str(STAGE3_RESULTS.relative_to(ROOT)), 'sha256': sha256_file(STAGE3_RESULTS)},
}
capacity_records = []
for row in rows:
    capacity_records.append({
        'capacity_pct': int(row['Capacity'].rstrip('%')), 'K': int(row['K']),
        'GBDT_mean': {'rescue_count': int(row['GBDT_mean rescue, n']), 'rescue_rate': float(row['GBDT_mean rescue, %'] / 100), 'overall_default_capture': float(row['GBDT_mean default capture, %'] / 100)},
        'FT_Transformer': {'rescue_count': int(row['FT-Transformer rescue, n']), 'rescue_rate': float(row['FT-Transformer rescue, %'] / 100), 'overall_default_capture': float(row['FT-Transformer default capture, %'] / 100)},
    })
result = {
    'experiment': 'Stage 9', 'version': 'V1', 'status': 'completed',
    'artifact_identities': artifact_identities, 'preflight': preflight, 'capacities': capacity_records,
    'blind_spot': {
        'source': 'accepted Stage 3 rule reconstructed from stage3_error_analysis_results_V1.json',
        'rule': 'target == 1 AND consensus_rank <= 0.50 AND rank_spread <= 0.10',
        'thresholds': {'consensus_rank_lte': deep_miss_threshold, 'rank_spread_lte': blind_spot_spread_threshold},
        'count': int(blind_spot.sum()), 'all_target_one': bool(np.all(target[blind_spot] == 1)),
        'blind_spot_cohort_sha256': blind_spot_cohort_sha256,
    },
    'blind_spot_cohort_sha256': blind_spot_cohort_sha256,
    'decomposition_at_30pct': {**{key.replace('-', '_'): value for key, value in decomposition.items()}, 'sum': BLIND_SPOT_N},
    'overall_default_count': default_count, 'primary_delta_rescue_30_pp': delta_rescue_30_pp,
    'decision': decision, 'final_test_used': False,
    'limitations': ['OOF random-CV не доказывает temporal stability.', 'Fixed capacity не является business policy.', 'Final test не использован.'],
}
summary = {key: result[key] for key in ('experiment', 'version', 'status', 'artifact_identities', 'preflight', 'capacities', 'blind_spot', 'blind_spot_cohort_sha256', 'decomposition_at_30pct', 'primary_delta_rescue_30_pp', 'decision', 'final_test_used', 'limitations')}
(GENERATED / 'stage9_rank_capacity_results_V1.json').write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(SUMMARY / 'stage9_rank_capacity_summary_V1.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Сохранены Stage 9 generated и summary JSON; final test не использован.')


Сохранены Stage 9 generated и summary JSON; final test не использован.


# Результат исследования

## ФАКТЫ

Stage 9 V1 завершён успешно. Нового обучения моделей не проводилось: анализ использует уже сохранённые OOF predictions.

Перед сравнением была проверена совместимость исходных artifacts:

- рабочая выборка содержит `289 614` наблюдений;
- `working_indices` Stage 3, Stage 7 и Stage 8 полностью совпадают;
- target между этапами полностью совпадает;
- SHA-256 рабочего индекса соответствует зафиксированному значению;
- все используемые probability arrays содержат конечные значения;
- сохранённый `GBDT_mean` из Stage 7 полностью совпадает с его копией в Stage 8.

Для анализа использована заранее определённая Stage 3 blind spot.

Поскольку отдельная row-level mask этой группы в Stage 3 artifact не сохранялась, cohort была детерминированно воспроизведена по принятым ещё в Stage 3 диагностическим границам: `target == 1`, `consensus_rank <= 0.50`, `rank_spread <= 0.10`. Повторного выбора границ в Stage 9 не проводилось.

Полученная cohort содержит ровно **805 дефолтных наблюдений**; её row-level identity дополнительно зафиксирована SHA-256:
`f0c227d1f07b922300b5488eb11c48590f0527e871802fb31f43e4020d8ba269`.

Сравнение проводилось не по threshold `0.5`, а при одинаковой доле компаний, отнесённых каждой моделью к high-risk.

| High-risk capacity | Компаний в top-K | GBDT_mean: blind-spot rescue | FT-Transformer: blind-spot rescue | GBDT_mean: захват всех дефолтов | FT-Transformer: захват всех дефолтов |
|---:|---:|---:|---:|---:|---:|
| 10% | 28 961 | 0 / 805 (0.000%) | 0 / 805 (0.000%) | 57.416% | 56.973% |
| 20% | 57 922 | 0 / 805 (0.000%) | 0 / 805 (0.000%) | 78.044% | 77.890% |
| 30% | 86 884 | 0 / 805 (0.000%) | 9 / 805 (1.118%) | 87.475% | 87.435% |

При основной capacity `30%`:

- обе модели одновременно не подняли в high-risk ни одного объекта blind spot;
- только FT-Transformer поднял `9` таких дефолтов;
- только `GBDT_mean` — `0`;
- остальные `796 из 805` не попали в top-30% ни у одной модели.

Таким образом:

**Δ blind-spot rescue при 30% capacity = +1.118 п.п. в пользу FT-Transformer.**

Заранее зафиксированный порог для подтверждения material rank complementarity составлял:

**+5.0 п.п.**

Фактический результат этот порог не достиг.

**Decision: `no_material_rank_complementarity`.**

Final test не использовался.

## ИНТЕРПРЕТАЦИЯ

FT-Transformer действительно ранжирует несколько трудных дефолтов иначе, чем `GBDT_mean`: при 30% capacity он поднял в high-risk 9 объектов Stage 3 blind spot, которые `GBDT_mean` туда не включил.

Однако масштаб этого эффекта очень небольшой:

**9 из 805 blind-spot дефолтов — около 1.1%.**

Этого недостаточно для заранее зафиксированного критерия material complementarity.

Более того, если смотреть не только на blind spot, а на все дефолты рабочей выборки, FT-Transformer не показывает преимущества при одинаковой capacity:

- при 10%: `56.973%` против `57.416%`;
- при 20%: `77.890%` против `78.044%`;
- при 30%: `87.435%` против `87.475%`.

То есть при одинаковом количестве компаний, которые модель может поместить в high-risk группу, FT-Transformer в целом не захватывает больше дефолтов, чем `GBDT_mean`.

Это помогает интерпретировать результат Stage 8.

Тогда FT-Transformer показывал более высокий Recall при диагностическом threshold `0.5`. Stage 9 не подтверждает, что этот рост Recall отражал существенное преимущество FT в ранжировании трудных дефолтов.

Поэтому повышенный Recall при `0.5` **не следует трактовать как подтверждённую model complementarity**.

При этом Stage 9 напрямую не исследовал calibration или форму шкалы вероятностей, поэтому утверждать, что разница Recall была доказанно вызвана только score scale, также нельзя.

Корректный итог:

**после устранения зависимости от общего числового threshold существенного rank-based преимущества FT-Transformer над `GBDT_mean` не обнаружено.**

## ОГРАНИЧЕНИЯ

- Анализ основан на OOF predictions из random CV. Он не доказывает temporal stability.
- Исследование относится к заранее определённой Stage 3 blind spot из `805` наблюдений и не характеризует все возможные типы ошибок моделей.
- Проверены только фиксированные capacity `10%`, `20%` и `30%`.
- Fixed capacity — исследовательский diagnostic, а не утверждённая business policy.
- Эксперимент проверяет различия в ranking, но напрямую не оценивает calibration или устройство шкалы probability score.
- Наличие 9 FT-only случаев показывает, что predictions моделей не полностью идентичны, но само по себе не доказывает полезность будущего ensemble.
- Final test не использовался.

## ВЫВОД

Исследовательский вопрос Stage 9 закрыт:

**FT-Transformer действительно по-другому ранжирует небольшое число трудных дефолтов, но при одинаковой high-risk capacity существенная model complementarity относительно `GBDT_mean` не подтверждена.**

Полученный эффект составляет только `+1.118 п.п.` blind-spot rescue при 30% capacity против заранее зафиксированного порога `+5.0 п.п.`.

Поэтому более высокий Recall FT-Transformer при threshold `0.5`, обнаруженный в Stage 8, не является достаточным evidence его преимущества над `GBDT_mean`.

## СЛЕДУЮЩИЙ ШАГ

Stage 9 показал, что FT-Transformer не даёт существенного rank-based преимущества над `GBDT_mean` на фиксированной Stage 3 blind spot.

При этом остаётся более общий вопрос.

За предыдущие этапы уже были проверены несколько разных моделей и способов их комбинации. По отдельности они не улучшили сильный `GBDT_mean`, но это ещё не означает, что их ошибки полностью совпадают.

Возможно, разные модели иногда правильно поднимают в high-risk **разные дефолтные компании**, даже если ни одна из моделей в целом не превосходит baseline.

### Следующий исследовательский вопрос

**Есть ли в уже сохранённых OOF-прогнозах принятых моделей существенный остаточный model reserve поверх `GBDT_mean`: существуют ли дефолты, которые `GBDT_mean` не поднимает в high-risk, но хотя бы одна из уже проверенных альтернативных моделей при той же risk capacity ранжирует высоко?**

Следующий этап должен проверить это **без нового обучения моделей**.

Для одинаковых фиксированных high-risk capacities нужно сравнить:

- сколько дефолтов захватывает сам `GBDT_mean`;
- сколько дефолтов могло бы быть захвачено, если считать успешным попадание объекта в high-risk хотя бы у одной из уже сохранённых альтернативных моделей.

Отдельно важно посмотреть:

- все дефолты рабочей выборки;
- фиксированные `805` дефолтов Stage 3 blind spot.

Такой `oracle`-анализ не является реальным ensemble и не показывает качество модели, которую можно использовать в работе. Он нужен только как диагностическая верхняя граница: **есть ли среди уже исследованных моделей дополнительный complementary signal вообще**.

Если прирост относительно `GBDT_mean` окажется небольшим, это будет означать, что заметного остаточного model reserve среди уже проверенных подходов не обнаружено.

Если прирост окажется существенным, появится конкретное основание отдельно исследовать leakage-safe способ объединения этих сигналов.

Final test для этого вопроса не используется.